# vec2text Colab LMI Legacy Dependencies

This notebook is designed for Google Colab and targets the heavier LMI path with an older, more conservative dependency stack.

It covers:
- environment setup
- pinning older Hugging Face dependencies for LMI
- LMI smoke test
- small-scale eval with inline Python code

Before running:
- switch Colab runtime to a stronger GPU if available
- enable High RAM if available
- if you want to use your own fork later, change `REPO_URL`


## 1. Check GPU

In Colab: `Runtime` -> `Change runtime type` -> prefer `A100 GPU` or `L4 GPU` and enable High RAM if available.

In [ ]:
import torch

print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu only')


## 2. Clone the repository and install dependencies

Default uses the public upstream repo so you do not need to push your own fork first.

In [ ]:
REPO_URL = 'https://github.com/jxmorris12/vec2text.git'
REPO_DIR = 'vec2text'


In [ ]:
!rm -rf "$REPO_DIR"
!git clone "$REPO_URL" "$REPO_DIR"


In [ ]:
%cd /content/vec2text
!python -m pip install --upgrade pip
!pip install -r requirements.txt
!pip install --upgrade --force-reinstall "transformers==4.38.2" "accelerate==0.27.2" "tokenizers==0.15.2" "sentence-transformers==2.6.1" "huggingface_hub==0.20.3"
!pip install -e . --no-deps


In [ ]:
import accelerate
import huggingface_hub
import sentence_transformers
import tokenizers
import transformers

print('transformers', transformers.__version__)
print('accelerate', accelerate.__version__)
print('tokenizers', tokenizers.__version__)
print('sentence-transformers', sentence_transformers.__version__)
print('huggingface_hub', huggingface_hub.__version__)


## 3. Runtime setup

In [ ]:
import os

os.environ['VEC2TEXT_CACHE'] = '/content/vec2text_cache'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.makedirs('/content/vec2text_cache', exist_ok=True)
os.makedirs('/content/vec2text/results', exist_ok=True)
print('VEC2TEXT_CACHE =', os.environ['VEC2TEXT_CACHE'])


## 3.1 Optional: load Hugging Face token from Colab Secrets

If you added a secret named `HF_TOKEN`, this cell will load it into the environment.
This helps with higher rate limits and gated model access, but is not required for all public models.

In [ ]:
import os

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        os.environ['HF_TOKEN'] = hf_token
        os.environ['HUGGINGFACE_HUB_TOKEN'] = hf_token
        print('HF_TOKEN loaded from Colab Secrets')
    else:
        print('HF_TOKEN not found in Colab Secrets')
except Exception as e:
    print('Could not load HF_TOKEN from Colab Secrets:', e)


## 3.2 Optional: mount Google Drive for saving results

If you want to keep outputs after the Colab session ends, run this cell.

In [ ]:
from pathlib import Path

DRIVE_SAVE_ENABLED = False
DRIVE_SAVE_DIR = '/content/drive/MyDrive/vec2text_results'

try:
    from google.colab import drive
    drive.mount('/content/drive')
    Path(DRIVE_SAVE_DIR).mkdir(parents=True, exist_ok=True)
    DRIVE_SAVE_ENABLED = True
    print('Google Drive mounted:', DRIVE_SAVE_DIR)
except Exception as e:
    print('Google Drive mount skipped or failed:', e)


## 4. Colab compatibility patches

The upstream repo currently needs two small runtime patches on Colab:
- add `import platform` in `experiments.py`
- disable `low_cpu_mem_usage` in `model_utils.py`
- disable `low_cpu_mem_usage` in `corrector_encoder.py`
- disable `low_cpu_mem_usage` in `api.py` model loading calls
- normalize dtype handling in `corrector_encoder.py` during generation
- provide a safe default `length_penalty` in `trainers/corrector.py`


In [ ]:
from pathlib import Path

# Patch 1: fix missing import in experiments.py
path1 = Path('/content/vec2text/vec2text/experiments.py')
text1 = path1.read_text(encoding='utf-8')
if 'import platform' not in text1:
    text1 = text1.replace('import os\n', 'import os\nimport platform\n', 1)
    path1.write_text(text1, encoding='utf-8')

# Patch 2: avoid meta-device loading issue in model_utils.py
path2 = Path('/content/vec2text/vec2text/models/model_utils.py')
text2 = path2.read_text(encoding='utf-8')
text2 = text2.replace('"low_cpu_mem_usage": True,', '"low_cpu_mem_usage": False,')
path2.write_text(text2, encoding='utf-8')

# Patch 3: avoid the same meta-device issue in corrector_encoder.py
path3 = Path('/content/vec2text/vec2text/models/corrector_encoder.py')
text3 = path3.read_text(encoding='utf-8')
old3 = """encoder_decoder = transformers.AutoModelForSeq2SeqLM.from_pretrained(\n            config.model_name_or_path\n        )"""
new3 = """encoder_decoder = transformers.AutoModelForSeq2SeqLM.from_pretrained(\n            config.model_name_or_path,\n            low_cpu_mem_usage=False,\n        )"""
if old3 in text3:
    text3 = text3.replace(old3, new3, 1)
    path3.write_text(text3, encoding='utf-8')

# Patch 4: avoid the same meta-device issue in api.py top-level from_pretrained calls
path4 = Path('/content/vec2text/vec2text/api.py')
text4 = path4.read_text(encoding='utf-8')
text4 = text4.replace(
    '        inversion_model = vec2text.models.InversionModel.from_pretrained(\n            "jxm/vec2text__openai_ada002__msmarco__msl128__hypothesizer"\n        )',
    '        inversion_model = vec2text.models.InversionModel.from_pretrained(\n            "jxm/vec2text__openai_ada002__msmarco__msl128__hypothesizer",\n            low_cpu_mem_usage=False,\n        )'
)
text4 = text4.replace(
    '        model = vec2text.models.CorrectorEncoderModel.from_pretrained(\n            "jxm/vec2text__openai_ada002__msmarco__msl128__corrector"\n        )',
    '        model = vec2text.models.CorrectorEncoderModel.from_pretrained(\n            "jxm/vec2text__openai_ada002__msmarco__msl128__corrector",\n            low_cpu_mem_usage=False,\n        )'
)
text4 = text4.replace(
    '        inversion_model = vec2text.models.InversionModel.from_pretrained(\n            "jxm/gtr__nq__32"\n        )',
    '        inversion_model = vec2text.models.InversionModel.from_pretrained(\n            "jxm/gtr__nq__32",\n            low_cpu_mem_usage=False,\n        )'
)
text4 = text4.replace(
    '        model = vec2text.models.CorrectorEncoderModel.from_pretrained(\n            "jxm/gtr__nq__32__correct"\n        )',
    '        model = vec2text.models.CorrectorEncoderModel.from_pretrained(\n            "jxm/gtr__nq__32__correct",\n            low_cpu_mem_usage=False,\n        )'
)
path4.write_text(text4, encoding='utf-8')

# Patch 5: fix dtype mismatch in corrector generation path
text3 = path3.read_text(encoding='utf-8')
old5 = """        batch_size, D = embedding.shape\n        assert embedding.shape == (batch_size, self.embedder_dim)\n        assert hypothesis_embedding.shape == (batch_size, self.embedder_dim)\n"""
new5 = """        batch_size, D = embedding.shape\n        assert embedding.shape == (batch_size, self.embedder_dim)\n        assert hypothesis_embedding.shape == (batch_size, self.embedder_dim)\n\n        transform_dtype = self.embedding_transform_1[0].weight.dtype\n        encoder_dtype = self.encoder_decoder.encoder.embed_tokens.weight.dtype\n        embedding = embedding.to(transform_dtype)\n        hypothesis_embedding = hypothesis_embedding.to(transform_dtype)\n"""
if old5 in text3:
    text3 = text3.replace(old5, new5, 1)

old5b = """        hypothesis_embedding = self.embedding_transform_3(hypothesis_embedding)\n        hypothesis_embedding = hypothesis_embedding.reshape(\n            (batch_size, self.num_repeat_tokens, self.encoder_hidden_dim)\n        )\n        inputs_embeds = self.encoder_decoder.encoder.embed_tokens(hypothesis_input_ids)\n"""
new5b = """        hypothesis_embedding = self.embedding_transform_3(hypothesis_embedding)\n        hypothesis_embedding = hypothesis_embedding.reshape(\n            (batch_size, self.num_repeat_tokens, self.encoder_hidden_dim)\n        )\n        embedding = embedding.to(encoder_dtype)\n        diff_embedding = diff_embedding.to(encoder_dtype)\n        hypothesis_embedding = hypothesis_embedding.to(encoder_dtype)\n        inputs_embeds = self.encoder_decoder.encoder.embed_tokens(hypothesis_input_ids).to(encoder_dtype)\n"""
if old5b in text3:
    text3 = text3.replace(old5b, new5b, 1)

old5c = """        sep_token = ones * self.encoder_decoder.config.eos_token_id\n        sep_token = self.encoder_decoder.encoder.embed_tokens(sep_token)\n"""
new5c = """        sep_token = ones * self.encoder_decoder.config.eos_token_id\n        sep_token = self.encoder_decoder.encoder.embed_tokens(sep_token).to(encoder_dtype)\n"""
if old5c in text3:
    text3 = text3.replace(old5c, new5c, 1)

old5d = """        if self.use_ln:\n            inputs_embeds = self.layernorm(inputs_embeds)\n"""
new5d = """        if self.use_ln:\n            self.layernorm = self.layernorm.to(device=inputs_embeds.device, dtype=encoder_dtype)\n            inputs_embeds = self.layernorm(inputs_embeds)\n"""
if old5d in text3:
    text3 = text3.replace(old5d, new5d, 1)

path3.write_text(text3, encoding='utf-8')

# Patch 6: guard against missing length_penalty in generation config
path5 = Path('/content/vec2text/vec2text/trainers/corrector.py')
text5 = path5.read_text(encoding='utf-8')
old6 = """            length_penalty = self.model.encoder_decoder.generation_config.length_penalty\n            output_length = (transition_scores < 0).sum(1)\n"""
new6 = """            length_penalty = self.model.encoder_decoder.generation_config.length_penalty\n            if length_penalty is None:\n                length_penalty = 1.0\n            output_length = (transition_scores < 0).sum(1)\n"""
if old6 in text5:
    text5 = text5.replace(old6, new6, 1)
path5.write_text(text5, encoding='utf-8')

print('patched experiments.py, model_utils.py, corrector_encoder.py, api.py, and trainers/corrector.py')


## 4.2 Runtime monkey patch for current Transformers / Accelerate behavior

Newer `transformers` versions can reject this repository's older nested loading path with a meta-device error.
This cell patches that runtime check for the current Colab session.

In [ ]:
import transformers
import transformers.modeling_utils as modeling_utils
import transformers.integrations.accelerate as tr_acc

def _patched_check_and_set_device_map(device_map):
    return device_map

tr_acc.check_and_set_device_map = _patched_check_and_set_device_map
modeling_utils.check_and_set_device_map = _patched_check_and_set_device_map

print('patched transformers meta-device check')


## 5. Import the library

In [ ]:
import vec2text

print('vec2text imported successfully')


## 5.1 Compatibility patch for tied-weights attributes

Newer `transformers` versions expect custom models to expose tied-weight metadata attributes.

In [ ]:
vec2text.models.InversionModel.all_tied_weights_keys = {}
vec2text.models.CorrectorEncoderModel.all_tied_weights_keys = {}
vec2text.models.InversionModel._tied_weights_keys = []
vec2text.models.CorrectorEncoderModel._tied_weights_keys = []

print('patched tied weight attributes')


## 5. `gtr-base` smoke test

In [ ]:
corrector = vec2text.load_pretrained_corrector('gtr-base')
print('Loaded pretrained corrector successfully.')
print(type(corrector).__name__)


In [ ]:
texts = [
    'Jack Morris is a PhD student at Cornell Tech in New York City',
    'It was the best of times, it was the worst of times'
]

outputs = vec2text.invert_strings(
    texts,
    corrector=corrector,
    num_steps=5,
)

for i, text in enumerate(outputs):
    print(f'[{i}] {text}')


In [ ]:
import json
from pathlib import Path

gtr_result = {
    'texts': texts,
    'outputs': outputs,
}

local_path = Path('results/gtr_outputs.json')
local_path.parent.mkdir(parents=True, exist_ok=True)
local_path.write_text(json.dumps(gtr_result, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved locally to', local_path)

if globals().get('DRIVE_SAVE_ENABLED'):
    drive_path = Path(DRIVE_SAVE_DIR) / 'gtr_outputs.json'
    drive_path.write_text(json.dumps(gtr_result, ensure_ascii=False, indent=2), encoding='utf-8')
    print('Saved to Google Drive at', drive_path)


## 6. LMI smoke test

This is heavier than `gtr-base`. If it fails, common causes are GPU RAM limits, HF auth requirements, or transient download failures.

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

MODEL_NAME = 'jxm/t5-base__llama-7b__one-million-instructions__emb'
CORRECTOR_NAME = 'jxm/t5-base___llama-7b___one-million-instructions__correct'

inversion_model = vec2text.models.InversionFromLogitsEmbModel.from_pretrained(MODEL_NAME).to(device)
corrector_model = vec2text.models.CorrectorEncoderFromLogitsModel.from_pretrained(CORRECTOR_NAME).to(device)
lmi_corrector = vec2text.load_corrector(inversion_model, corrector_model)
print('Loaded inversion model and corrector successfully.')
print(type(lmi_corrector).__name__)


## 7. Small-scale eval

This notebook uses inline Python instead of relying on a local helper script.

In [ ]:
import argparse
import json
from pathlib import Path

import vec2text

alias = 'jxm/t5-base__llama-7b__one-million-instructions__emb'
dataset = 'python_code_alpaca'
num_samples = 10
batch_size = 1
use_less_data = 32
output_json = 'results/minimal_eval_lmi_10.json'

experiment, trainer = vec2text.analyze_utils.load_experiment_and_trainer_from_pretrained(
    alias,
    use_less_data=use_less_data,
)
trainer.model.use_frozen_embeddings_as_input = False
trainer.args.per_device_eval_batch_size = batch_size

available = sorted(trainer.eval_dataset.keys())
if dataset not in trainer.eval_dataset:
    raise KeyError(f"Dataset {dataset!r} not found. Available eval datasets: {available}")

eval_dataset = trainer.eval_dataset[dataset]
if 'frozen_embeddings' in eval_dataset.column_names:
    eval_dataset = eval_dataset.remove_columns('frozen_embeddings')
eval_dataset = eval_dataset.select(range(min(num_samples, len(eval_dataset))))

metrics = trainer.evaluate(eval_dataset=eval_dataset)
reduced_metrics = {
    'alias': alias,
    'dataset': dataset,
    'num_samples': len(eval_dataset),
    'available_eval_datasets': available,
    'eval_bleu_score': metrics.get('eval_bleu_score'),
    'eval_bleu_score_sem': metrics.get('eval_bleu_score_sem'),
    'eval_exact_match': metrics.get('eval_exact_match'),
    'eval_exact_match_sem': metrics.get('eval_exact_match_sem'),
    'eval_accuracy': metrics.get('eval_accuracy'),
    'eval_perplexity': metrics.get('eval_perplexity'),
    'eval_loss': metrics.get('eval_loss'),
    'eval_runtime': metrics.get('eval_runtime'),
    'eval_samples_per_second': metrics.get('eval_samples_per_second'),
}

output_path = Path(output_json)
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(json.dumps(reduced_metrics, indent=2), encoding='utf-8')

print(json.dumps(reduced_metrics, indent=2))


In [ ]:
from pathlib import Path
import json

if globals().get('DRIVE_SAVE_ENABLED'):
    drive_eval_path = Path(DRIVE_SAVE_DIR) / 'minimal_eval_lmi_10.json'
    drive_eval_path.write_text(json.dumps(reduced_metrics, ensure_ascii=False, indent=2), encoding='utf-8')
    print('Saved eval metrics to Google Drive at', drive_eval_path)
else:
    print('Google Drive not mounted; eval metrics already saved locally to', output_json)


In [ ]:
import json
from pathlib import Path

for path in [Path('results/minimal_eval_lmi_10.json')]:
    if path.exists():
        print('\n===', path, '===')
        print(json.dumps(json.loads(path.read_text(encoding='utf-8')), indent=2))


## 8. Optional: larger eval

Run this only after the 10-sample eval succeeds.

In [ ]:
# Change these values only after the smaller run works.
# num_samples = 20
# batch_size = 2
# use_less_data = 64
